In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
adata = sc.read_h5ad("../data/processed/Filbin_step3.h5ad")

In [3]:





print(f"shape: {adata.shape}        (anchor: 2458 x 8494)")
print(f"layers: {list(adata.layers.keys())}   (anchor: tpm, centered)")

print("\nPer-patient counts:")
print(adata.obs["patient"].value_counts())
print("(anchors: MUV5 708, BCH836 527, BCH869 492, BCH1126 299, MUV10 286, MUV1 146)")

X = np.asarray(adata.X)
print(f"\n.X mean           = {X.mean():.4f}   (anchor: 1.428)")
print(f".X zero fraction  = {(X == 0).mean():.4f}   (anchor: 0.544)")
print(f".X range          = [{X.min():.2f}, {X.max():.2f}]   (anchor: [0, 15.15])")

Er = np.asarray(adata.layers["centered"])
print(f"\ncentered mean    = {Er.mean():.2e}   (anchor: ~1e-8)")
print(f"centered std     = {Er.std():.4f}   (anchor: ~1.64)")
print(f"centered range   = [{Er.min():.2f}, {Er.max():.2f}]   (anchor: ~[-9.3, +15.1])")

print(f"\nEa min           = {adata.var['Ea'].min():.4f}   (anchor: >= 4.0)")

shape: (2458, 8494)        (anchor: 2458 x 8494)
layers: ['centered', 'tpm']   (anchor: tpm, centered)

Per-patient counts:
MUV5       708
BCH836     527
BCH869     492
BCH1126    299
MUV10      286
MUV1       146
Name: patient, dtype: int64
(anchors: MUV5 708, BCH836 527, BCH869 492, BCH1126 299, MUV10 286, MUV1 146)

.X mean           = 1.4282   (anchor: 1.428)
.X zero fraction  = 0.5439   (anchor: 0.544)
.X range          = [0.00, 15.15]   (anchor: [0, 15.15])

centered mean    = 1.19e-08   (anchor: ~1e-8)
centered std     = 1.6411   (anchor: ~1.64)
centered range   = [-9.32, 15.10]   (anchor: ~[-9.3, +15.1])

Ea min           = 4.0002   (anchor: >= 4.0)


In [4]:
# Cell-cell Pearson on per-tumor centered expression


Er = np.asarray(adata.layers["centered"])   # (2458, 8494), float32
print(f"input: shape {Er.shape}, dtype {Er.dtype}")

# np.corrcoef: rows are variables -> rows of Er are cells -> output is cell-cell
C = np.corrcoef(Er).astype(np.float32)
print(f"correlation matrix: shape {C.shape}, dtype {C.dtype}, size {C.nbytes / 1e6:.1f} MB")

# Mechanical sanity
print(f"\nsymmetry  max |C - C.T| : {np.max(np.abs(C - C.T)):.2e}   (expect ~0)")
print(f"diagonal  min / max     : {np.diag(C).min():.6f} / {np.diag(C).max():.6f}   (expect 1.0)")
iu = np.triu_indices_from(C, k=1)
off = C[iu]
print(f"off-diag  range         : [{off.min():.3f}, {off.max():.3f}]")
print(f"off-diag  median / mean : {np.median(off):+.4f} / {off.mean():+.4f}")

input: shape (2458, 8494), dtype float32
correlation matrix: shape (2458, 2458), dtype float32, size 24.2 MB

symmetry  max |C - C.T| : 0.00e+00   (expect ~0)
diagonal  min / max     : 1.000000 / 1.000000   (expect 1.0)
off-diag  range         : [-0.157, 0.970]
off-diag  median / mean : -0.0005 / -0.0003


In [5]:
# Within-tumor vs cross-tumor correlation distribution
patients = adata.obs["patient"].to_numpy()
same_tumor = patients[:, None] == patients[None, :]

corr_vals = C[iu]
same_mask = same_tumor[iu]
within = corr_vals[same_mask]
cross  = corr_vals[~same_mask]

print(f"\nWithin-tumor pairs ({len(within):>9,}):  "
      f"median {np.median(within):+.4f}  mean {within.mean():+.4f}  std {within.std():.4f}")
print(f"Cross-tumor pairs  ({len(cross):>9,}):  "
      f"median {np.median(cross):+.4f}  mean {cross.mean():+.4f}  std {cross.std():.4f}")
print(f"Mean(within) - Mean(cross) = {within.mean() - cross.mean():+.4f}")


Within-tumor pairs (  605,556):  median -0.0024  mean -0.0018  std 0.0482
Cross-tumor pairs  (2,414,097):  median -0.0002  mean +0.0001  std 0.0296
Mean(within) - Mean(cross) = -0.0019
